<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 🗺️ EarthDaily Agriculture - Field Level Maps (FLM) Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

In [ ]:
print(manager.sfd_list.columns)

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 - Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from flm_extractor.py**

### 🗺️ Configure extraction

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor
cov_extractor = CoverageExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Initialize coverage extractor for getting image IDs
coverage_extractor = CoverageExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config
)

coverage_extractor.setup_coverage_parameters(
    vegetation_index='LAI',
    start_date="2025-08-01",
    use_specific_date=False, #(False =lastest image, True = at this date)
    clear_cover_min=100,
    delay=3,
    mask='ML',
    partial_frequency=50,
    exclude_columns=[]
)

from earthdaily.agriculture.extractors.FLM_functions import FLMExtractor
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor

# Initialize FLM extractor
flm_extractor = FLMExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config
)

# Setup FLM parameters
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    map_format= None,  # Options: 'png', 'tiff.zip', 'shp.zip' None
    postprocess='stats', # Options: 'stats', 'file', 'links'
    output_epsg=4326,
    skip_existing=True,
    output_path=manager.output_result_dir,
)

In [ ]:
test_entity = {
    "id": "test_001",
    "geometry": "POLYGON ((-97.70066562 37.14062335, -97.69927729 37.14227539, -97.69935777 37.14233954, -97.70004188 37.14248389, -97.70008212 37.14359058, -97.698633420000007 37.14495386, -97.69867367 37.14554728, -97.698512700000009 37.145868050000004, -97.69823101 37.14591616, -97.69804992 37.14739167, -97.69760727 37.14755205, -97.69285876 37.14771243, -97.69261731 37.14753601, -97.692536830000009 37.14062335, -97.69265756 37.140495030000004, -97.69778835 37.14041483, -97.698170650000009 37.14059127, -97.69839198 37.14046295, -97.70048454 37.14044691, -97.70066562 37.14062335))",
    "crop":"WINTER_WHEAT",
    "name":"toto"
}

image_id ="sentinel-2-c1-l2a|S2B_T14SPG_20251010T172020_L2A"

### 🗺️ API call

In [ ]:
print("\n--- Test: get_flm_map ---")
try:
    raw_response = flm_extractor.get_flm_map(test_entity, image_id)
    print("✅ Raw API response received")
    print(f"Status Code: {raw_response.status_code}")
    print(f"Content Type: {raw_response.headers.get('Content-Type')}")
    print(f"Content Length: {len(raw_response.content)} bytes")
    
    # Try to parse as JSON if it's a JSON response
    try:
        json_data = raw_response.json()
        print("\n📄 JSON Response Preview:")
        print(json_data)
    except:
        print("\n📦 Binary Response (not JSON)")
        print(f"First 100 bytes: {raw_response.content[:100]}")
        
except Exception as e:
    print(f"❌ API call failed: {e}")
    import traceback
    traceback.print_exc()

### 🗺️ API call safe

In [ ]:
print("\n--- Test: get_flm_safe ---")
safe_result = flm_extractor.get_flm_map_safe(test_entity,image_id)
print(safe_result)

#### Test format_flm_json

In [ ]:
print("\n--- Test: format_flm_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = flm_extractor.format_flm_map_stats_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_coverage_json: No valid data from API.")

#### Test format_flm_file

In [ ]:
print("\n--- Test: format_flm_map_file ---")
try:
    # Use the raw_response from the previous cell
    result = flm_extractor.format_flm_map_file(
        entity_data=test_entity,
        image_id=image_id,
        response=raw_response,
        output_path=None  # Will use default from flm_params
    )
    
    print("✅ File formatting completed")
    print(f"\nStatus: {result.get('status')}")
    
    if result.get('status') == 'downloaded':
        print(f"Entity ID: {result.get('entity_id')}")
        print(f"Image ID: {result.get('image_id')}")
        print(f"Map Format: {result.get('map_format')}")
        print(f"File Count: {result.get('file_count')}")
        print(f"Total Size: {result.get('total_size_bytes'):,} bytes")
        print(f"\n📁 Saved Files:")
        for file_path in result.get('saved_files', []):
            print(f"  - {file_path}")
            # Verify file exists
            if os.path.exists(file_path):
                file_size = os.path.getsize(file_path)
                print(f"    ✓ File exists ({file_size:,} bytes)")
            else:
                print(f"    ✗ File not found!")
    
    elif result.get('status') == 'error':
        print(f"❌ Error: {result.get('error_message')}")
        print(f"Entity ID: {result.get('entity_id')}")
        print(f"Map Format: {result.get('map_format')}")
    
except Exception as e:
    print(f"❌ File formatting failed: {e}")
    import traceback
    traceback.print_exc()

# Optional: Display image if PNG format
if result.get('status') == 'downloaded' and result.get('map_format') == 'png':
    try:
        from PIL import Image
        import matplotlib.pyplot as plt
        
        png_file = result.get('saved_files', [])[0]
        img = Image.open(png_file)
        
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.title(f"FLM Map: {result.get('entity_id')}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        
        print(f"\n📊 Image dimensions: {img.size[0]} x {img.size[1]} pixels")
    except ImportError:
        print("\n💡 Install PIL/Pillow to display images: pip install pillow")
    except Exception as e:
        print(f"\n⚠️ Could not display image: {e}")

### 🗺️ process_single_seasonfield

Extract stats

In [ ]:

# Import your function
import pandas as pd

# Test entity
row = pd.Series({
    "id": "toto",
    "name":"Test_Field",
    "geometry": "POLYGON ((-97.70066562 37.14062335, -97.69927729 37.14227539, -97.69935777 37.14233954, -97.70004188 37.14248389, -97.70008212 37.14359058, -97.698633420000007 37.14495386, -97.69867367 37.14554728, -97.698512700000009 37.145868050000004, -97.69823101 37.14591616, -97.69804992 37.14739167, -97.69760727 37.14755205, -97.69285876 37.14771243, -97.69261731 37.14753601, -97.692536830000009 37.14062335, -97.69265756 37.140495030000004, -97.69778835 37.14041483, -97.698170650000009 37.14059127, -97.69839198 37.14046295, -97.70048454 37.14044691, -97.70066562 37.14062335))",
    "image_id" :"sentinel-2-c1-l2a|S2B_T14SPG_20251010T172020_L2A"
})

result=flm_extractor.process_single_entity_flm(row)


Extract file 

In [ ]:
# Setup FLM parameters
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    map_format= "png",  # Options: 'png', 'tiff.zip', 'shp.zip' None
    postprocess="file",
    output_epsg=4326,
    skip_existing=True,
    output_path=manager.output_result_dir,
)
result=flm_extractor.process_single_entity_flm(row)

In [ ]:
# Setup FLM parameters
flm_extractor.setup_flm_parameters(
    vegetation_index='COLORCOMPOSITION',
    map_format= 'png',  # Options: 'png', 'tiff.zip', 'shp.zip' None
    postprocess="file",
    output_epsg=4326,
    skip_existing=True,
    output_path=manager.output_result_dir,
    directLinks= False,
)
result=flm_extractor.process_single_entity_flm(row)

In [ ]:
print(result)

Extract histogram

`postprocess='histogram'` returns a flat DataFrame combining the global stats (`stat_min`, `stat_max`, `stat_mean`) with per-bucket values (`value_min`, `value_max`, `num_pixels`, `area`). The query automatically sets `histogram=true` against the API.

In [ ]:
# Setup FLM parameters in histogram mode
# number_bins is optional (1..255). When set, appends '&numberOfBins=<value>' to the API URL.
# legendType is optional ('Fixed', 'Dynamic', 'Common'). When set, appends '&legendType=<value>'.
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='histogram',
    output_epsg=4326,
    skip_existing=True,
    number_bins=15,
    legendType='Fixed',
)

result_histogram = flm_extractor.process_single_entity_flm(row)
print(f"Error: {result_histogram.get('error')}")
if result_histogram.get('data') is not None:
    print(f"Data shape: {result_histogram['data'].shape}")
    display(result_histogram['data'])

## **🧪 Step 4: Showcase — every postprocess mode**

`postprocess` accepts four modes. The cells below run a single entity through each one
and display the resulting DataFrame so you can compare the columns produced.

| Mode | Returns | Key params |
|---|---|---|
| `stats` | `stat_min`, `stat_mean`, `stat_max` (1 row) | — |
| `links` | direct PNG / worldfile / thumbnail URLs + bbox + worldFile coefficients | auto-sets `directLinks=True`, `map_format` must be `None` |
| `file` | downloads raster to disk, returns one summary row | requires `map_format` (`png` / `tiff.zip` / `shp.zip`) and `output_path` |
| `histogram` | global stats + per-bucket `value_min_*`, `value_max_*`, `num_pixels_*`, `area_*` | optional `number_bins` (1–255), `legendType` |

All cells reuse the `row` Series and `image_id` defined above.

### 4.1 — `postprocess='stats'`

In [ ]:
# Mode: stats — single-row DataFrame with min/mean/max of the FLM
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='stats',
    output_epsg=4326,
)

result_stats = flm_extractor.process_single_entity_flm(row)
print(f"Error: {result_stats.get('error')}")
if result_stats.get('data') is not None:
    print(f"Shape: {result_stats['data'].shape}")
    print(f"Columns: {list(result_stats['data'].columns)}")
    display(result_stats['data'])

### 4.2 — `postprocess='links'`

Returns direct download URLs (PNG, worldfile, thumbnail) plus the worldFile transform
and bounding box, so you can render or georeference the raster on the client side
without saving it locally. `directLinks` is auto-corrected to `True` and `map_format` must be `None`.

In [ ]:
# Mode: links — useful when a downstream service (web app, GIS) will fetch the PNG itself
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='links',
    map_format=None,           # required by 'links' mode
    output_epsg=4326,
)

result_links = flm_extractor.process_single_entity_flm(row)
print(f"Error: {result_links.get('error')}")
if result_links.get('data') is not None:
    df = result_links['data']
    print(f"Shape: {df.shape}")
    # Show the link columns in a transposed view for readability
    link_cols = [c for c in df.columns if 'link' in c or c.startswith('bbox_') or c.startswith('worldfile_') or c.startswith('map_')]
    display(df[link_cols].T)

### 4.3 — `postprocess='file'` with `map_format='png'`

Downloads the raster as a PNG into `output_path`. The returned DataFrame summarises the save
(status, file count, total bytes, saved file paths).

In [ ]:
# Mode: file (png) — quickest format, no georeferencing
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='file',
    map_format='png',
    output_path=manager.output_result_dir,
    output_epsg=4326,
    skip_existing=False,
)

result_file_png = flm_extractor.process_single_entity_flm(row)
print(f"Error: {result_file_png.get('error')}")
if result_file_png.get('data') is not None:
    display(result_file_png['data'])

### 4.4 — `postprocess='file'` with `map_format='tiff.zip'`

Downloads a zipped GeoTIFF and extracts the `.tif` to `output_path`. Use this when you
need a georeferenced raster for GIS tools (QGIS, rasterio, gdal).

In [ ]:
# Mode: file (tiff.zip) — georeferenced raster, ready for rasterio / GIS tools
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='file',
    map_format='tiff.zip',
    output_path=manager.output_result_dir,
    output_epsg=4326,
    skip_existing=False,
)

result_file_tiff = flm_extractor.process_single_entity_flm(row)
print(f"Error: {result_file_tiff.get('error')}")
if result_file_tiff.get('data') is not None:
    display(result_file_tiff['data'])

### 4.5 — `postprocess='file'` with `map_format='shp.zip'`

Downloads a zipped Shapefile and extracts every component (`.shp`, `.shx`, `.dbf`, `.prj`, …)
to `output_path`. Use this when downstream consumers need vector geometry rather than a raster.

In [ ]:
# Mode: file (shp.zip) — vector output, all shapefile components are extracted
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='file',
    map_format='shp.zip',
    output_path=manager.output_result_dir,
    output_epsg=4326,
    skip_existing=False,
)

result_file_shp = flm_extractor.process_single_entity_flm(row)
print(f"Error: {result_file_shp.get('error')}")
if result_file_shp.get('data') is not None:
    display(result_file_shp['data'])

### 4.6 — `postprocess='histogram'` (recap)

Same as cell 32 above but kept here for completeness — a single-row DataFrame with
global stats plus per-bucket columns. Tweak `number_bins` and `legendType` to control
how the API discretises the histogram.

In [ ]:
# Mode: histogram — global stats + per-bucket distribution in one wide row
flm_extractor.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='histogram',
    output_epsg=4326,
    number_bins=10,           # optional: 1..255
    legendType='Dynamic',     # optional: 'Fixed' / 'Dynamic' / 'Common'
)

result_hist = flm_extractor.process_single_entity_flm(row)
print(f"Error: {result_hist.get('error')}")
if result_hist.get('data') is not None:
    df = result_hist['data']
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    display(df)

### 4.7 — Side-by-side comparison

Compares the column shape produced by each mode so you can pick the right one for
your downstream pipeline.

In [ ]:
import pandas as pd

comparison = []
for label, res in [
    ('stats', result_stats),
    ('links', result_links),
    ('file (png)', result_file_png),
    ('file (tiff.zip)', result_file_tiff),
    ('file (shp.zip)', result_file_shp),
    ('histogram', result_hist),
]:
    df = res.get('data')
    comparison.append({
        'mode': label,
        'rows': 0 if df is None else len(df),
        'cols': 0 if df is None else df.shape[1],
        'error': res.get('error'),
        'sample_columns': '' if df is None else ', '.join(list(df.columns)[:6]) + ('…' if df is not None and df.shape[1] > 6 else ''),
    })

display(pd.DataFrame(comparison))

### 🗺️ process_flm_bulk_extraction_parallel

In [ ]:
# Extract coverage and keep geometry in output

from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor
cov_extractor = CoverageExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id"}

cov_extractor.setup_coverage_parameters(
                                    vegetation_index= 'NDVI',
                                    start_date="2025-01-01",
                                    clear_cover_min=80,
                                    use_specific_date=False,
                                    filter="duplicate",
                                    delay=3,
                                    mask='ML',#Available masks (auto, native, ACM, ML). To configure.
                                    partial_frequency=20,
                                    exclude_columns=[],
                                    column_mapping=column_mapping) 

top500 = manager.sfd_list.head(10)

results = cov_extractor.process_entity_coverage_bulk_parallel(
    entity_list=top500,
    params=None,          # or pass overrides here
    max_workers=20,        # adjust threads depending on API rate limits
    output_path=manager.output_result_dir,
    fail_safe= False
)

# Initialize FLM extractor
from earthdaily.agriculture.extractors.FLM_functions import FLMExtractor
flm_extractor_bulk = FLMExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )
flm_extractor_bulk = FLMExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config
)

# Setup FLM parameters
flm_extractor_bulk.setup_flm_parameters(
    vegetation_index='NDVI',
    map_format=None,  # Options: 'png', 'tiff.zip', 'shp.zip' None
    postprocess='stats',
    output_epsg=4326,
    skip_existing=True,
    output_path=manager.output_result_dir,
)

result_flm = flm_extractor_bulk.process_entity_flm_bulk_parallel(
    entity_list=results['results_df'],
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column=None,
    filter_value=None,
    filter_type="exclude", # filter type used to 'include' or 'exclude' row matching column and value filter
    merge_existing='auto',
    skip_export=False,
    prefix="flm_stat")

In [ ]:
flm_input = results['results_df']
print(flm_input.columns)
print(len(flm_input))


In [ ]:
# Initialize coverage extractor for getting image IDs
flm_extractor = flm_extractor.process_entity_flm_bulk_parallel(
    entity_list=flm_input,
    params=None,          # or pass overrides here
    max_workers=30,        # adjust threads depending on API rate limits
    output_path=manager.output_result_dir,
    partial_frequency=50,
    fail_safe= False
)

In [ ]:
# Get the clean DataFrame
if 'results' in locals():
    results_df = results.get('results', [])
    if results_df:
        print(f"Total results: {len(results_df)}")
        print(results_df.columns)

## **🧪 Step 5: Bulk extraction — every postprocess mode**

Re-runs each postprocess mode through `process_entity_flm_bulk_parallel` against the
small `flm_input` DataFrame produced by the coverage step above. Use a fresh
`FLMExtractor` per mode so results from earlier modes are not overwritten.

### 5.1 — Bulk `stats`

In [ ]:
from earthdaily.agriculture.extractors.FLM_functions import FLMExtractor

flm_bulk_stats = FLMExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
)
flm_bulk_stats.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='stats',
    output_epsg=4326,
)

result_bulk_stats = flm_bulk_stats.process_entity_flm_bulk_parallel(
    entity_list=flm_input,
    max_workers=10,
    output_path=manager.output_result_dir,
    skip_export=True,
    prefix='flm_stats',
)

print(f"✅ {result_bulk_stats['successful_calculations']}/{result_bulk_stats['total_calculations']} succeeded")
display(result_bulk_stats['results_df'].head())

### 5.2 — Bulk `links`

In [ ]:
flm_bulk_links = FLMExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
)
flm_bulk_links.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='links',
    map_format=None,
    output_epsg=4326,
)

result_bulk_links = flm_bulk_links.process_entity_flm_bulk_parallel(
    entity_list=flm_input,
    max_workers=10,
    output_path=manager.output_result_dir,
    skip_export=True,
    prefix='flm_links',
)

print(f"✅ {result_bulk_links['successful_calculations']}/{result_bulk_links['total_calculations']} succeeded")
display(result_bulk_links['results_df'].head())

### 5.3 — Bulk `file` (PNG)

Downloads one PNG per entity into `output_path`. The aggregated DataFrame holds the
save status; the actual rasters land on disk.

In [ ]:
flm_bulk_file = FLMExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
)
flm_bulk_file.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='file',
    map_format='png',
    output_path=manager.output_result_dir,
    output_epsg=4326,
    skip_existing=False,
)

result_bulk_file = flm_bulk_file.process_entity_flm_bulk_parallel(
    entity_list=flm_input,
    max_workers=5,
    output_path=manager.output_result_dir,
    skip_export=True,
    prefix='flm_file_png',
)

print(f"✅ {result_bulk_file['successful_calculations']}/{result_bulk_file['total_calculations']} succeeded")
display(result_bulk_file['results_df'].head())

### 5.4 — Bulk `histogram`

In [ ]:
flm_bulk_hist = FLMExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
)
flm_bulk_hist.setup_flm_parameters(
    vegetation_index='NDVI',
    postprocess='histogram',
    output_epsg=4326,
    number_bins=10,
)

result_bulk_hist = flm_bulk_hist.process_entity_flm_bulk_parallel(
    entity_list=flm_input,
    max_workers=10,
    output_path=manager.output_result_dir,
    skip_export=True,
    prefix='flm_histogram',
)

print(f"✅ {result_bulk_hist['successful_calculations']}/{result_bulk_hist['total_calculations']} succeeded")
display(result_bulk_hist['results_df'].head())

### 5.5 — Bulk run summary

One-line summary of every bulk mode for quick comparison.

In [ ]:
import pandas as pd

bulk_summary = []
for label, res in [
    ('stats', result_bulk_stats),
    ('links', result_bulk_links),
    ('file (png)', result_bulk_file),
    ('histogram', result_bulk_hist),
]:
    df = res.get('results_df')
    bulk_summary.append({
        'mode': label,
        'total': res.get('total_calculations'),
        'success': res.get('successful_calculations'),
        'failed': res.get('failed_calculations'),
        'cols': 0 if df is None else df.shape[1],
    })

display(pd.DataFrame(bulk_summary))